In [ ]:
#pip install openai pandas openpyxl python-dotenv

In [ ]:
import os
import re
import json
import time
import uuid
import logging
import threading
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI, RateLimitError
 

In [ ]:
logger = logging.getLogger("pipeline")
logger.setLevel(logging.INFO)
 
_console = logging.StreamHandler()
_console.setFormatter(logging.Formatter(
    "%(asctime)s [%(threadName)s] %(message)s", datefmt="%H:%M:%S"
))
logger.addHandler(_console)

In [ ]:
# ==============================================================================
# API KEYS & CLIENT ROTATION
# ==============================================================================
# CHANGE: Keys loaded from environment variables instead of hardcoded.
# Create a .env file:
#   NVAPI_KEY_1=nvapi-...
#   NVAPI_KEY_2=nvapi-...
#   NVAPI_KEY_3=nvapi-...
# Install: pip install python-dotenv
# ==============================================================================
 
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
 
API_KEYS = [ "", "", 
]


KEY_NAMES = ["KEY_1", "KEY_2", "KEY_3"]
 
_key_index      = 0
_key_index_lock = threading.Lock()
 
 
def get_client(key_index: int) -> OpenAI:
    return OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=API_KEYS[key_index]
    )
 
 
def next_key_index(current: int) -> int:
    return (current + 1) % len(API_KEYS)

In [ ]:
LINKAGE_EXCEL_PATH = r"BRSR_final_verified_sorted.xlsx"
FIRMS_FOLDER       = r"FY_2023_2024"
OUTPUT_FOLDER      = r"output_files"
STATUS_LOG_PATH    = r"logs/status_log.xlsx"
 
RLM_MODEL         = "meta/llama-3.3-70b-instruct"
BATCH_WRITE_EVERY = 20
MAX_WORKERS       = 5
 
# CHANGE: Character budget for context assembly (~3000 tokens).
# Replaces the blind 300-row cutoff that could overflow the model's context.
CONTEXT_CHAR_BUDGET = 12000
 
os.makedirs(OUTPUT_FOLDER,                    exist_ok=True)
os.makedirs(os.path.dirname(STATUS_LOG_PATH), exist_ok=True)
 
RUN_ID = str(uuid.uuid4())[:8]
 
OUTPUT_COLUMNS = [
    "file_name", "firm_name", "financial_year", "brsr_code",
    "gri_no", "gri_sub_no", "section_no", "sub_section_no",
    "gri_standard", "gri_substandard", "gri_header", "gri_sub_header",
    "section_header", "section", "sub_section_header", "sub_section",
    "linkage", "actual_text",
    "evaluation", "present_elements", "missing_elements", "evidence_quotes",
    "present_elements_actual", "missing_elements_actual",
    "confidence", "total_num", "present_num", "missing_num", "score",
    "length_evidence_quotes"
]
 
SUMMARY_COLUMNS = [
    "run_id", "firm_id", "firm_file", "start_time", "end_time",
    "duration_sec", "total_rows_expected", "processed_rows", "status",
    "output_path", "last_active_key", "retries_total", "rate_limit_events",
    "input_tokens_total", "output_tokens_total", "total_tokens",
    "error_message_last"
]
 
EVENTS_COLUMNS = [
    "run_id", "timestamp", "firm_id", "row_index", "action",
    "key_from", "key_to", "wait_seconds", "input_tokens", "output_tokens",
    "exception_type", "exception_message", "notes"
]

In [ ]:
_log_lock      = threading.Lock()
_event_buffer  = []
_summary_cache = None
_events_cache  = None

In [ ]:
def _ensure_log_loaded():
    """Load summary & events from disk into memory — runs once."""
    global _summary_cache, _events_cache
    if _summary_cache is not None:
        return
    if os.path.exists(STATUS_LOG_PATH):
        try:
            _summary_cache = pd.read_excel(STATUS_LOG_PATH, sheet_name="summary")
            _events_cache  = pd.read_excel(STATUS_LOG_PATH, sheet_name="events")
            for col in SUMMARY_COLUMNS:
                if col not in _summary_cache.columns:
                    _summary_cache[col] = None
            for col in EVENTS_COLUMNS:
                if col not in _events_cache.columns:
                    _events_cache[col] = None
            _summary_cache = _summary_cache[SUMMARY_COLUMNS]
            _events_cache  = _events_cache[EVENTS_COLUMNS]
            return
        except Exception:
            pass
    _summary_cache = pd.DataFrame(columns=SUMMARY_COLUMNS)
    _events_cache  = pd.DataFrame(columns=EVENTS_COLUMNS)
    

In [ ]:
def _flush_log():
    """Write in-memory state to Excel. Must be called under _log_lock."""
    global _event_buffer, _events_cache
    _ensure_log_loaded()
    if _event_buffer:
        _events_cache = pd.concat(
            [_events_cache, pd.DataFrame(_event_buffer)], ignore_index=True
        )
        _event_buffer = []
    tmp = STATUS_LOG_PATH + ".tmp"
    with pd.ExcelWriter(tmp, engine="openpyxl") as writer:
        _summary_cache.to_excel(writer, sheet_name="summary", index=False)
        _events_cache.to_excel(writer,  sheet_name="events",  index=False)
    os.replace(tmp, STATUS_LOG_PATH)

In [ ]:
def _empty_event_row() -> dict:
    return {col: None for col in EVENTS_COLUMNS}
 
 
def log_event(firm_id: str, action: str, row_index=None, key_from=None,
              key_to=None, wait_seconds=None, input_tokens=None,
              output_tokens=None, exception_type=None,
              exception_message=None, notes=None):
    """Append event to in-memory buffer — zero disk I/O."""
    row = {
        **_empty_event_row(),
        "run_id":            RUN_ID,
        "timestamp":         datetime.now().isoformat(timespec="seconds"),
        "firm_id":           firm_id,
        "row_index":         row_index,
        "action":            action,
        "key_from":          key_from,
        "key_to":            key_to,
        "wait_seconds":      wait_seconds,
        "input_tokens":      input_tokens,
        "output_tokens":     output_tokens,
        "exception_type":    exception_type,
        "exception_message": exception_message,
        "notes":             notes,
    }
    with _log_lock:
        _event_buffer.append(row)

In [ ]:
def flush_log():
    """Public: flush buffered events to disk. Call once per firm."""
    with _log_lock:
        _flush_log()

In [ ]:
def log_summary_start(firm_id: str, firm_file: str,
                      total_rows_expected: int, output_path: str):
    with _log_lock:
        _ensure_log_loaded()
        global _summary_cache
        mask = ((_summary_cache["run_id"] == RUN_ID) &
                (_summary_cache["firm_id"] == firm_id))
        row = {
            "run_id":              RUN_ID,
            "firm_id":             firm_id,
            "firm_file":           firm_file,
            "start_time":          datetime.now().isoformat(timespec="seconds"),
            "end_time":            None,
            "duration_sec":        None,
            "total_rows_expected": total_rows_expected,
            "processed_rows":      0,
            "status":              "in_progress",
            "output_path":         output_path,
            "last_active_key":     KEY_NAMES[_key_index],
            "retries_total":       0,
            "rate_limit_events":   0,
            "input_tokens_total":  0,
            "output_tokens_total": 0,
            "total_tokens":        0,
            "error_message_last":  None,
        }
        if mask.any():
            for k, v in row.items():
                _summary_cache.loc[mask, k] = v
        else:
            _summary_cache = pd.concat(
                [_summary_cache, pd.DataFrame([row])], ignore_index=True
            )

In [ ]:
def log_summary_update(firm_id: str, **kwargs):
    with _log_lock:
        _ensure_log_loaded()
        global _summary_cache
        mask = ((_summary_cache["run_id"] == RUN_ID) &
                (_summary_cache["firm_id"] == firm_id))
        if not mask.any():
            return
        for k, v in kwargs.items():
            if k in SUMMARY_COLUMNS:
                _summary_cache.loc[mask, k] = v

In [ ]:
def log_summary_complete(firm_id: str, start_time_str: str,
                         processed_rows: int, last_active_key: str,
                         retries_total: int, rate_limit_events: int,
                         input_tokens_total: int, output_tokens_total: int,
                         error_message: str = None):
    end_time = datetime.now()
    try:
        duration = round(
            (end_time - datetime.fromisoformat(start_time_str)).total_seconds(), 1
        )
    except Exception:
        duration = None
 
    status = "error" if error_message else "completed"
    with _log_lock:
        _ensure_log_loaded()
        global _summary_cache
        mask = ((_summary_cache["run_id"] == RUN_ID) &
                (_summary_cache["firm_id"] == firm_id))
        if not mask.any():
            return
        _summary_cache.loc[mask, "end_time"]            = end_time.isoformat(timespec="seconds")
        _summary_cache.loc[mask, "duration_sec"]        = duration
        _summary_cache.loc[mask, "processed_rows"]      = processed_rows
        _summary_cache.loc[mask, "status"]              = status
        _summary_cache.loc[mask, "last_active_key"]     = last_active_key
        _summary_cache.loc[mask, "retries_total"]       = retries_total
        _summary_cache.loc[mask, "rate_limit_events"]   = rate_limit_events
        _summary_cache.loc[mask, "input_tokens_total"]  = input_tokens_total
        _summary_cache.loc[mask, "output_tokens_total"] = output_tokens_total
        _summary_cache.loc[mask, "total_tokens"]        = input_tokens_total + output_tokens_total
        _summary_cache.loc[mask, "error_message_last"]  = error_message
        # flush at firm completion — the natural checkpoint
        _flush_log()

In [ ]:
def log_skip(firm_id: str, firm_file: str, output_path: str,
             total_rows_expected: int):
    now = datetime.now().isoformat(timespec="seconds")
    with _log_lock:
        _ensure_log_loaded()
        global _summary_cache
        mask = ((_summary_cache["run_id"] == RUN_ID) &
                (_summary_cache["firm_id"] == firm_id))
        if not mask.any():
            row = {
                "run_id":              RUN_ID,
                "firm_id":             firm_id,
                "firm_file":           firm_file,
                "start_time":          now,
                "end_time":            now,
                "duration_sec":        0,
                "total_rows_expected": total_rows_expected,
                "processed_rows":      total_rows_expected,
                "status":              "skipped_already_evaluated",
                "output_path":         output_path,
                "last_active_key":     None,
                "retries_total":       0,
                "rate_limit_events":   0,
                "input_tokens_total":  0,
                "output_tokens_total": 0,
                "total_tokens":        0,
                "error_message_last":  None,
            }
            _summary_cache = pd.concat(
                [_summary_cache, pd.DataFrame([row])], ignore_index=True
            )
        _event_buffer.append({
            **_empty_event_row(),
            "run_id":    RUN_ID,
            "timestamp": now,
            "firm_id":   firm_id,
            "action":    "skip_already_evaluated",
            "notes":     f"output already complete at {output_path}",
        })
        _flush_log()
 
 
def is_firm_already_completed(firm_id: str, expected_row_count: int,
                               output_path: str) -> bool:
    with _log_lock:
        _ensure_log_loaded()
        try:
            firm_rows = _summary_cache[_summary_cache["firm_id"] == firm_id]
            if not firm_rows.empty:
                latest = firm_rows.iloc[-1]
                if (str(latest.get("status", "")) == "completed" and
                        int(latest.get("processed_rows", 0)) >= expected_row_count):
                    return True
        except Exception:
            pass
 
    if os.path.exists(output_path):
        try:
            if len(pd.read_excel(output_path)) >= expected_row_count:
                return True
        except Exception:
            pass
 
    return False

In [ ]:
SYSTEM_PROMPT = """
You are an ESG reporting evaluation assistant.
 
STRICT RULES:
 
1. Use ONLY the Firm Disclosure Context and the GRI actual_text requirement provided.
 
2. The actual_text contains multiple consolidated GRI sub-requirements
   (e.g., "a. report its legal name; c. report the location of its headquarters;").
   You MUST split actual_text into individual sub-requirements and evaluate EACH one separately.
 
3. present_elements: List ONLY the sub-requirement texts from actual_text that the firm HAS disclosed.
   Example: if firm reported legal name → add "a. report its legal name;" to present_elements.
 
4. missing_elements: List ONLY the sub-requirement texts from actual_text that the firm has NOT
   disclosed or insufficiently disclosed.
 
5. evidence_quotes: Exact verbatim quotes copied from the Firm Disclosure Data.
   Do NOT paraphrase or fabricate. Only include text that literally appears in the data.
 
6. present_elements_actual: Exact excerpts from Firm Disclosure Data that confirm each present
   element. No fabrication. Only what literally appears in the data.
   Example: "NameOfCompany: TATA STEEL LIMITED"
 
7. missing_elements_actual: Describe what required disclosures are verifiably absent from the
   Firm Disclosure Data. No fabrication.
   Example: "No headquarters location found in firm disclosure data."
 
8. DO NOT put firm disclosure text in present_elements or missing_elements.
   Those fields must ONLY contain sub-requirement texts from actual_text.
 
9. Interpretation rules:
   meets           = all sub-requirements from actual_text clearly present in firm disclosure
   partial         = some sub-requirements present, some missing
   does_not_meet   = all or most core sub-requirements missing from firm disclosure
   not_applicable  = disclosure requirement explicitly not relevant to this firm
 
10. confidence = CERTAINTY IN YOUR VERDICT (not amount of evidence found):
    meets + clear evidence found           → 0.8 to 1.0
    partial + mixed disclosure             → 0.5 to 0.8
    does_not_meet + clearly absent         → 0.7 to 1.0
    any verdict + ambiguous/unclear data   → 0.3 to 0.5
    A does_not_meet with nothing found is HIGH confidence (0.8–1.0).
    NEVER assign 0.0 just because no evidence was found.
 
11. Output STRICT JSON ONLY — no preamble, no explanation, no markdown fences:
{
  "evaluation": "",
  "present_elements": [],
  "missing_elements": [],
  "evidence_quotes": [],
  "present_elements_actual": [],
  "missing_elements_actual": [],
  "confidence": 0.0
}
"""
 
 
def build_user_prompt(firm_name, year, gri_standard, gri_substandard,
                      actual_text, section, sub_section, firm_context):
    return f"""
Firm Name: {firm_name}
Financial Year: {year}
 
GRI Standard: {gri_standard}
GRI Sub-Standard: {gri_substandard}
Section: {section}
Sub-Section: {sub_section}
 
GRI REQUIREMENT (actual_text) — contains multiple consolidated sub-requirements:
{actual_text}
 
Firm Disclosure Data (relevant rows):
{firm_context}
 
TASK:
1. Split the actual_text into individual sub-requirements.
2. For each sub-requirement, check if the firm has disclosed it in the Firm Disclosure Data.
3. Put the sub-requirement text in present_elements if disclosed.
4. Put the sub-requirement text in missing_elements if NOT disclosed.
5. Put exact verbatim quotes from firm disclosure data in evidence_quotes.
6. In present_elements_actual, list exact firm data excerpts confirming each present element.
7. In missing_elements_actual, describe what required disclosures are verifiably absent.
8. Assign overall evaluation: meets / partial / does_not_meet / not_applicable.
9. Assign confidence as certainty in your verdict (NOT amount of evidence found).
Return STRICT JSON only.
"""

In [ ]:
def count_sub_requirements(actual_text: str) -> int:
    text  = actual_text.replace('•', ';')
    parts = [p.strip() for p in text.split(';') if p.strip()]
    return len(parts)
 
 
def build_firm_context(firm_df: pd.DataFrame, sub_section: str,
                       combined_col: pd.Series) -> str:
    """
    CHANGE (OPTIMIZATION #2 — context building):
 
    OLD approach — called ~300 times per firm, each call did:
      mask = pd.concat(
          [col.str.contains(sub_section) for col in str_cache.values()],
          axis=1
      ).any(axis=1)
      → Created N boolean Series (one per column), concatenated into a
        DataFrame, then reduced. For a firm with 20 columns and 5,000 rows,
        that's 20 Series + 1 DataFrame created and discarded PER CALL.
        Over 300 calls: 6,000 temporary Series + 300 temporary DataFrames.
 
    NEW approach:
      - combined_col is a single pre-computed Series: each cell contains all
        column values joined into one string (computed once on main thread)
      - Filtering is now ONE .str.contains() call on ONE Series
      - Same results, ~20x less pandas object creation
 
    Also replaces the arbitrary 300-row cutoff with CONTEXT_CHAR_BUDGET
    to prevent context window overflow on firms with long narrative rows.
    """
    filtered = pd.DataFrame()
    if sub_section:
        mask     = combined_col.str.contains(
            re.escape(sub_section), case=False, na=False
        )
        filtered = firm_df[mask]
 
    if filtered.empty:
        filtered = firm_df
 
    # character-budget-aware assembly
    lines     = []
    char_used = 0
    for _, row in filtered.iterrows():
        line = " | ".join(f"{c}: {row[c]}" for c in filtered.columns)
        if char_used + len(line) > CONTEXT_CHAR_BUDGET:
            break
        lines.append(line)
        char_used += len(line)
 
    return "\n".join(lines)

In [ ]:
def _extract_json_object(text: str) -> dict:
    """
    CHANGE (OPTIMIZATION #6 — robust JSON parsing):
 
    OLD approach: text[text.find("{") : text.rfind("}") + 1]
      → rfind("}") finds the LAST } in the entire output. If a string value
        inside the JSON contains "}", this grabs too much text and json.loads
        fails, or worse, silently parses wrong.
 
    NEW approach:
      1. Try json.loads() on the full cleaned text (fast path)
      2. Fall back to depth-counted brace matching that respects string literals
         — correctly handles } inside "quoted strings"
    """
    if not text:
        raise ValueError("Empty model output")
    text = text.strip()
    text = re.sub(r"```json", "", text, flags=re.I)
    text = re.sub(r"```",     "", text)
    text = text.strip()
 
    # fast path — entire cleaned text is valid JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
 
    # depth-counted brace matching (respects string boundaries)
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found in model output")
 
    depth       = 0
    in_string   = False
    escape_next = False
 
    for i in range(start, len(text)):
        ch = text[i]
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"' and not escape_next:
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
 
    raise ValueError("No complete JSON object found in model output")

In [ ]:
def evaluate_llm(user_prompt: str, firm_id: str,
                 row_index: int, max_tries: int = 3):
    global _key_index
 
    last_err         = None
    retries          = 0
    rate_limit_count = 0
 
    for attempt in range(1, max_tries + 1):
 
        consecutive_rl = 0
 
        while True:
            with _key_index_lock:
                active_index = _key_index
                active_name  = KEY_NAMES[active_index]
 
            try:
                completion = get_client(active_index).chat.completions.create(
                    model=RLM_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_prompt}
                    ],
                    temperature=0.2,
                    top_p=0.7,
                    max_tokens=1024,
                    stream=False
                )
 
                raw_text      = completion.choices[0].message.content or ""
                input_tokens  = completion.usage.prompt_tokens     if completion.usage else 0
                output_tokens = completion.usage.completion_tokens if completion.usage else 0
                retries      += attempt - 1
                break
 
            except RateLimitError as exc:
                rate_limit_count += 1
                consecutive_rl   += 1
 
                log_event(
                    firm_id=firm_id, action="rate_limited",
                    row_index=row_index, key_from=active_name,
                    exception_type=type(exc).__name__,
                    exception_message=str(exc)[:200],
                    notes=f"attempt {attempt}"
                )
                logger.warning(
                    "Rate limit on %s (row %s)", active_name, row_index
                )
 
                if consecutive_rl >= len(API_KEYS):
                    log_event(
                        firm_id=firm_id, action="global_wait_60s",
                        row_index=row_index, wait_seconds=60,
                        notes="all keys rate-limited"
                    )
                    logger.warning("All keys rate-limited — waiting 60s...")
                    time.sleep(60)
                    consecutive_rl = 0
                    log_event(
                        firm_id=firm_id, action="resume_after_wait",
                        row_index=row_index,
                        notes="resuming after 60s global wait"
                    )
 
                with _key_index_lock:
                    old_index  = _key_index
                    _key_index = next_key_index(_key_index)
                    new_index  = _key_index
 
                old_name     = KEY_NAMES[old_index]
                new_name     = KEY_NAMES[new_index]
                action_taken = f"switched_to_{new_name}"
 
                log_event(
                    firm_id=firm_id, action="switch_key",
                    row_index=row_index,
                    key_from=old_name, key_to=new_name,
                    notes=action_taken
                )
                logger.info("%s (row %s)", action_taken, row_index)
 
        # parse & validate
        try:
            data = _extract_json_object(raw_text)
        except Exception as exc:
            last_err = exc
            logger.warning("Attempt %d parse failed: %s", attempt, exc)
            user_prompt += (
                "\n\nIMPORTANT: Output ONLY valid JSON in the required "
                "schema. No extra text, no markdown."
            )
            continue
 
        data.setdefault("evaluation",              None)
        data.setdefault("present_elements",        [])
        data.setdefault("missing_elements",        [])
        data.setdefault("evidence_quotes",         [])
        data.setdefault("present_elements_actual", [])
        data.setdefault("missing_elements_actual", [])
        data.setdefault("confidence",              0.0)
 
        for field in ("present_elements", "missing_elements",
                      "evidence_quotes",  "present_elements_actual",
                      "missing_elements_actual"):
            if not isinstance(data[field], list):
                data[field] = []
 
        try:
            data["confidence"] = float(data["confidence"])
        except Exception:
            data["confidence"] = 0.0
 
        eval_val = data.get("evaluation")
        while isinstance(eval_val, dict):
            eval_val = next(iter(eval_val.values()), None)
        if isinstance(eval_val, list):
            eval_val = eval_val[0] if eval_val else None
        if eval_val is not None:
            eval_val = str(eval_val).strip()
        data["evaluation"] = eval_val
 
        return data, input_tokens, output_tokens, retries, rate_limit_count
 
    raise RuntimeError(
        f"LLM failed after {max_tries} tries. Last error: {last_err}"
    )

In [ ]:
def process_row(lr, firm_df, firm_name, year, brsr_code,
                file_name, combined_col, firm_id, row_index):
    """
    CHANGE: Accepts combined_col (pd.Series) instead of str_cache (dict).
    Passed through to build_firm_context() for single-column filtering.
    """
    gri_sub_no      = str(lr.get("gri_sub_no",      "")).strip()
    sub_section_no  = str(lr.get("sub_section_no",  "")).strip()
    gri_standard    = str(lr.get("gri_standard",    "")).strip()
    gri_substandard = str(lr.get("gri_substandard", "")).strip()
    actual_text     = str(lr.get("actual_text",     "")).strip()
    section         = str(lr.get("section",         "")).strip()
    sub_section     = str(lr.get("sub_section",     "")).strip()
 
    key = (firm_name, year, gri_sub_no, sub_section_no)
 
    # fast-path: no actual_text → not_applicable, no LLM call
    if not actual_text:
        eq_json    = json.dumps([])
        output_row = {col: lr.get(col, "") for col in lr.index}
        output_row.update({
            "file_name":               file_name,
            "firm_name":               firm_name,
            "financial_year":          year,
            "brsr_code":               brsr_code,
            "evaluation":              "not_applicable",
            "present_elements":        json.dumps([]),
            "missing_elements":        json.dumps([]),
            "evidence_quotes":         eq_json,
            "present_elements_actual": json.dumps([]),
            "missing_elements_actual": json.dumps([]),
            "confidence":              0.0,
            "total_num":               0,
            "present_num":             0,
            "missing_num":             0,
            "score":                   None,
            "length_evidence_quotes":  len(eq_json),
        })
        return (key,
                {col: output_row.get(col, "") for col in OUTPUT_COLUMNS},
                0, 0, 0, 0)
 
    firm_context = build_firm_context(firm_df, sub_section, combined_col)
    user_prompt  = build_user_prompt(
        firm_name, year, gri_standard, gri_substandard,
        actual_text, section, sub_section, firm_context
    )
 
    logger.info("Evaluating : %s | %s | %s...",
                gri_sub_no, sub_section_no, gri_standard[:45])
 
    try:
        res, in_tok, out_tok, retries, rl_count = evaluate_llm(
            user_prompt, firm_id=firm_id, row_index=row_index
        )
    except RuntimeError as exc:
        logger.error("FAILED: %s — marking as error", exc)
        res      = {
            "evaluation":              "error",
            "present_elements":        [],
            "missing_elements":        [],
            "evidence_quotes":         [],
            "present_elements_actual": [],
            "missing_elements_actual": [],
            "confidence":              0.0,
        }
        in_tok   = 0
        out_tok  = 0
        retries  = 0
        rl_count = 0
 
    present_num = len(res.get("present_elements", []))
    missing_num = len(res.get("missing_elements", []))
    total_num   = count_sub_requirements(actual_text)
 
    # CHANGE: cap score at 1.0 — prevents LLM splitting sub-requirements
    # differently than count_sub_requirements(), which could yield score > 1.0
    score = min(round(present_num / total_num, 4), 1.0) if total_num > 0 else None
 
    eq_json = json.dumps(
        res.get("evidence_quotes", []), ensure_ascii=False
    )
 
    output_row = {col: lr.get(col, "") for col in lr.index}
    output_row.update({
        "file_name":               file_name,
        "firm_name":               firm_name,
        "financial_year":          year,
        "brsr_code":               brsr_code,
        "evaluation":              res.get("evaluation"),
        "present_elements":        json.dumps(
            res.get("present_elements",        []), ensure_ascii=False),
        "missing_elements":        json.dumps(
            res.get("missing_elements",        []), ensure_ascii=False),
        "evidence_quotes":         eq_json,
        "present_elements_actual": json.dumps(
            res.get("present_elements_actual", []), ensure_ascii=False),
        "missing_elements_actual": json.dumps(
            res.get("missing_elements_actual", []), ensure_ascii=False),
        "confidence":              res.get("confidence", 0.0),
        "total_num":               total_num,
        "present_num":             present_num,
        "missing_num":             missing_num,
        "score":                   score,
        "length_evidence_quotes":  len(eq_json),
    })
    return (key,
            {col: output_row.get(col, "") for col in OUTPUT_COLUMNS},
            in_tok, out_tok, retries, rl_count)

In [ ]:
logger.info("Run ID  : %s", RUN_ID)
logger.info("Log     : %s", STATUS_LOG_PATH)
logger.info("Starting evaluation pipeline...\n")
 
linkage_df         = pd.read_excel(LINKAGE_EXCEL_PATH).fillna("")
expected_row_count = len(linkage_df)
logger.info("Linkage rows loaded      : %d", expected_row_count)
 
# CHANGE: pre-compute sub-requirement counts once — same for every firm,
# no need to call count_sub_requirements() inside each worker.
_sub_req_counts = {
    i: count_sub_requirements(str(row.get("actual_text", "")))
    for i, (_, row) in enumerate(linkage_df.iterrows())
}
 
all_csv_files = sorted([
    f for f in os.listdir(FIRMS_FOLDER) if f.endswith(".csv")
])
logger.info("Firm CSV files found     : %d\n", len(all_csv_files))
 
for file in all_csv_files:
 
    path      = os.path.join(FIRMS_FOLDER, file)
    file_name = file
    base      = file.replace(".csv", "")
    firm_id   = base
 
    output_filename = base.replace("XBRL", "GRI").replace("_xbrl", "_GRI") + ".xlsx"
    output_path     = os.path.join(OUTPUT_FOLDER, output_filename)
 
    year_match = re.search(r"(20\d{2}).*(20\d{2})", base)
    year       = (f"FY {year_match.group(1)}-{year_match.group(2)}"
                  if year_match else "UNKNOWN_YEAR")
    firm_name  = re.sub(r"(20\d{2}).*(20\d{2})", "", base).replace("_", " ").strip()
    firm_name  = re.sub(r"xbrl.*", "", firm_name, flags=re.I).strip()
    brsr_code  = (f"{firm_name}_{year.replace('FY ', '')}"
                  if year != "UNKNOWN_YEAR" else f"{firm_name}_UNKNOWN")
 
    logger.info("=" * 60)
    logger.info("Firm     : %s | %s", firm_name, year)
    logger.info("Input    : %s", file)
    logger.info("Output   : %s", output_filename)
 
    # ── skip if already fully evaluated ──────────────────────────────────────
    if is_firm_already_completed(firm_id, expected_row_count, output_path):
        logger.info("[SKIP] Already complete. Skipping.\n")
        log_skip(firm_id, file, output_path, expected_row_count)
        continue
 
    # ── load existing output for resume detection ─────────────────────────────
    if os.path.exists(output_path):
        out_df_existing = pd.read_excel(output_path).fillna("")
        logger.info("  Resuming  : %d/%d rows done",
                     len(out_df_existing), expected_row_count)
    else:
        out_df_existing = pd.DataFrame()
 
    completed = set()
    if not out_df_existing.empty:
        required_cols = {"firm_name", "financial_year",
                         "gri_sub_no", "sub_section_no"}
        if required_cols.issubset(set(out_df_existing.columns)):
            completed = set(zip(
                out_df_existing["firm_name"],
                out_df_existing["financial_year"],
                out_df_existing["gri_sub_no"].astype(str),
                out_df_existing["sub_section_no"].astype(str)
            ))
        else:
            logger.info("  Old schema detected — starting fresh.")
            out_df_existing = pd.DataFrame()
 
    pending = [
        (i, lr)
        for i, (_, lr) in enumerate(linkage_df.iterrows())
        if (firm_name, year,
            str(lr.get("gri_sub_no",     "")).strip(),
            str(lr.get("sub_section_no", "")).strip()) not in completed
    ]
 
    if not pending:
        logger.info("  All rows already evaluated — skipping.\n")
        continue
 
    logger.info("  Pending   : %d rows\n", len(pending))
 
    log_summary_update(firm_id=firm_id, status="pending")
 
    # ── load firm CSV + pre-compute combined text column ──────────────────────
    # CHANGE (OPTIMIZATION #2):
    # OLD: str_cache = {col: firm_df[col].astype(str) for col in firm_df.columns}
    #   → dict of Series passed to every worker, each call iterated all columns
    # NEW: single Series joining all column values per row, computed once here
    firm_df      = pd.read_csv(path).fillna("")
    combined_col = firm_df.apply(
        lambda r: " | ".join(f"{c}: {r[c]}" for c in firm_df.columns), axis=1
    )
 
    start_time_str = datetime.now().isoformat(timespec="seconds")
    log_summary_start(firm_id, file, expected_row_count, output_path)
    log_event(
        firm_id=firm_id, action="start_firm",
        notes=f"{len(pending)} pending rows | run_id={RUN_ID}"
    )
 
    results         = []
    done_count      = 0
    firm_in_tokens  = 0
    firm_out_tokens = 0
    firm_retries    = 0
    firm_rl_events  = 0
 
    # ── thread pool ───────────────────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                process_row,
                lr, firm_df, firm_name, year, brsr_code,
                file_name, combined_col, firm_id, row_index
            ): (row_index, lr)
            for row_index, lr in pending
        }
 
        for future in as_completed(futures):
            try:
                (key, output_row,
                 in_tok, out_tok,
                 retries, rl_count) = future.result()
            except Exception as exc:
                logger.error("Unexpected worker error: %s", exc)
                log_event(
                    firm_id=firm_id, action="error",
                    exception_type=type(exc).__name__,
                    exception_message=str(exc)[:300]
                )
                continue
 
            results.append(output_row)
            completed.add(key)
            done_count      += 1
            firm_in_tokens  += in_tok
            firm_out_tokens += out_tok
            firm_retries    += retries
            firm_rl_events  += rl_count
 
            # CHANGE (OPTIMIZATION #3 — deferred concat):
            # OLD: pd.concat([out_df_existing, pd.DataFrame(results)]) every 20 rows
            #   → rebuilds growing DataFrame each time = O(n²) total write volume
            # NEW: write ONLY new results to a .partial checkpoint file
            #   → out_df_existing merge happens once at final flush
            if done_count % BATCH_WRITE_EVERY == 0:
                pd.DataFrame(results).to_excel(
                    output_path + ".partial", index=False
                )
                log_event(
                    firm_id=firm_id, action="row_completed",
                    row_index=done_count,
                    input_tokens=firm_in_tokens,
                    output_tokens=firm_out_tokens,
                    notes=f"{done_count} rows completed so far"
                )
                logger.info("  Batch checkpoint : %d rows done", done_count)
 
    # ── final flush: SINGLE merge with out_df_existing ────────────────────────
    if results:
        final_df = pd.concat(
            [out_df_existing, pd.DataFrame(results)],
            ignore_index=True
        )
        final_df.to_excel(output_path, index=False)
    else:
        final_df = out_df_existing
 
    # clean up partial checkpoint
    partial_path = output_path + ".partial"
    if os.path.exists(partial_path):
        os.remove(partial_path)
 
    final_count      = len(final_df)
    remaining_errors = int(
        (final_df["evaluation"].astype(str).str.strip() == "error").sum()
    ) if not final_df.empty else 0
 
    if final_count != expected_row_count:
        logger.warning("  WARNING: final rows %d != expected %d",
                        final_count, expected_row_count)
 
    error_msg = (
        f"{remaining_errors} rows still error"
        if remaining_errors > 0 else None
    )
 
    with _key_index_lock:
        active_key_name = KEY_NAMES[_key_index]
 
    log_summary_complete(
        firm_id=firm_id,
        start_time_str=start_time_str,
        processed_rows=final_count,
        last_active_key=active_key_name,
        retries_total=firm_retries,
        rate_limit_events=firm_rl_events,
        input_tokens_total=firm_in_tokens,
        output_tokens_total=firm_out_tokens,
        error_message=error_msg
    )
    log_event(
        firm_id=firm_id, action="completed_firm",
        input_tokens=firm_in_tokens,
        output_tokens=firm_out_tokens,
        notes=(f"final_rows={final_count} "
               f"remaining_errors={remaining_errors} "
               f"tokens={firm_in_tokens + firm_out_tokens}")
    )
 
    # flush all buffered log events to disk — once per firm
    flush_log()
 
    logger.info("")
    logger.info("  DONE — %s", output_path)
    logger.info("  Rows       : %d/%d", final_count, expected_row_count)
    logger.info("  Errors     : %d", remaining_errors)
    logger.info("  Tokens     : %d in / %d out", firm_in_tokens, firm_out_tokens)
    logger.info("  RL events  : %d | Retries: %d\n", firm_rl_events, firm_retries)
 
# final safety flush
flush_log()
 
logger.info("=" * 60)
logger.info("ALL FIRMS EVALUATED")
logger.info("Outputs : %s", OUTPUT_FOLDER)
logger.info("Log     : %s", STATUS_LOG_PATH)
logger.info("=" * 60)